# Customer Analytics Walkthrough

This notebook provides a narrative walkthrough of the privacy-safe public refactor of the Bento Customer Analytics project. The original operational system used private POS/order data; all executable examples here use **synthetic data only**.

The goal is to connect business questions to analytical methods, add uncertainty estimates, and demonstrate leakage-aware temporal validation rather than presenting dashboard metrics as if they were predictive models.

## Questions addressed

1. How recently and how often do customers purchase?
2. What is a typical repurchase interval, and how uncertain is that estimate?
3. Which products are purchased together, and how stable is the observed lift?
4. Can historical customer behaviour help rank the probability of a purchase in the following 30 days?
5. How does model performance change across future time periods?

In [ ]:
from pathlib import Path
import subprocess, sys
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from src.analytics import load_orders, load_order_items, rfm_analysis, repurchase_analysis, market_basket_rules
from src.modeling import bootstrap_median_ci, bootstrap_rule_lift_ci, build_repurchase_snapshot_dataset, walk_forward_logistic_evaluation

orders_path = ROOT / 'sample_data' / 'synthetic_orders.csv'
items_path = ROOT / 'sample_data' / 'synthetic_order_items.csv'
if not orders_path.exists() or not items_path.exists():
    subprocess.run([sys.executable, str(ROOT / 'scripts' / 'generate_synthetic_data.py')], check=True)

orders = load_orders(orders_path)
items = load_order_items(items_path)
orders.head()

## 1. RFM as an interpretable descriptive baseline

RFM is intentionally treated as an operational segmentation rule rather than a statistically estimated model. It is useful because the segments are transparent and easy for a business user to act on, but the thresholds should not be interpreted as universal.

In [ ]:
rfm = rfm_analysis(orders, reference_date='2025-06-30')
rfm['segment'].value_counts().to_frame('customers')

## 2. Repurchase interval with bootstrap uncertainty

A point estimate alone can look more precise than the data justify. The public extension therefore bootstraps the customer-level median repurchase interval.

**Synthetic demo result:** median = **8.5 days**, 95% bootstrap CI = **[7.0, 11.0] days**.

In [ ]:
repurchase = repurchase_analysis(orders)
repurchase_ci = bootstrap_median_ci(repurchase['median_gap_days'], n_boot=5000)
repurchase_ci

In [ ]:
repurchase['median_gap_days'].plot(kind='hist', bins=15, title='Synthetic customer repurchase intervals')
plt.xlabel('Median gap between purchase dates (days)')
plt.show()

## 3. Market-basket association with an order-level bootstrap

The dashboard originally reported support, confidence and lift. The extension adds an order-level bootstrap interval so the transaction/basket structure is preserved when uncertainty is estimated.

**Synthetic demo result:** Grilled Chicken Bento → Tea Egg has lift **3.58**, 95% bootstrap CI **[3.05, 4.23]**.

This does **not** imply causation; it only describes co-purchase association in the generated demo.

In [ ]:
rules = market_basket_rules(orders, items, min_support=0.015)
top_rule = rules.iloc[0]
lift_ci = bootstrap_rule_lift_ci(
    orders,
    items,
    antecedent=top_rule['antecedent'],
    consequent=top_rule['consequent'],
    n_boot=5000,
)
top_rule.to_frame('value'), lift_ci

## 4. Future 30-day repurchase model

The predictive extension asks a different question from the descriptive dashboard: **given information available at a snapshot date, will the customer place at least one valid order in the following 30 days?**

Features are computed only from the past: recency, 90-day frequency/spend, lifetime orders/spend, average order value and customer tenure. The model is logistic regression, chosen for interpretability and as a strong statistical baseline.

In [ ]:
snapshots = ['2025-02-28', '2025-03-31', '2025-04-30', '2025-05-31']
model_data = build_repurchase_snapshot_dataset(orders, snapshots, horizon_days=30)
model_data.head()

## 5. Expanding-window temporal validation

A random row split would let future periods influence training and would be a weak validation design for a time-dependent purchase task. Instead, each month is tested using only earlier snapshots for training.

| Held-out snapshot | ROC-AUC | Average Precision | Brier score |
|---|---:|---:|---:|
| 2025-03-31 | 0.796 | 0.749 | 0.197 |
| 2025-04-30 | 0.929 | 0.577 | 0.097 |
| 2025-05-31 | 0.774 | 0.591 | 0.324 |
| **Mean** | **0.833** | **0.639** | **0.206** |

The final month is especially useful: ROC-AUC remains reasonable, but Brier score deteriorates as the positive-class rate changes. That gap illustrates why **ranking discrimination and probability calibration should be evaluated separately**.

In [ ]:
evaluation, model = walk_forward_logistic_evaluation(model_data, horizon_days=30)
evaluation

In [ ]:
plot_df = evaluation.set_index('test_snapshot')[['roc_auc', 'average_precision', 'brier_score']]
plot_df.plot(marker='o', title='Walk-forward performance on synthetic holdouts')
plt.ylim(0, 1)
plt.ylabel('Metric value')
plt.show()

## Interpretation and limitations

- The notebook demonstrates a **methodological workflow**, not verified commercial predictive performance.
- The public model is fitted to synthetic data because the operational customer-level dataset is private.
- Customers can appear in multiple historical snapshots; temporal splitting blocks future-to-past leakage but does not make repeated snapshots statistically independent.
- The changing Brier score shows why calibration monitoring matters when behaviour or base rates shift.
- A larger follow-up study could compare logistic regression with survival analysis, use customer-grouped sensitivity checks, and validate on a genuinely out-of-sample operational period under an approved privacy arrangement.

## Takeaway

The original project began as a business dashboard. The public refactor now separates three layers clearly: **descriptive analytics** (RFM, basket and repurchase summaries), **uncertainty quantification** (bootstrap intervals), and **predictive evaluation** (future repurchase with temporal holdouts). That separation makes the analytical claims easier to audit and harder to overstate.